## 🧠 MICE Process Flow (Visual Diagram)

      ┌────────────────────────────┐
      │     Original Dataset       │
      │ (Some missing values exist)│
      └────────────┬───────────────┘
                   │
                   ▼
      ┌────────────────────────────┐
      │  Step 1: Initial Imputation │
      │  (Fill all missing values   │
      │   with mean/median/random)  │
      └────────────┬───────────────┘
                   │
                   ▼
    ┌───────────────────────────────┐
    │ Step 2: Select 1 variable with │
    │ missing data (e.g., Age)       │
    │ Predict it using other columns │
    └────────────┬──────────────────┘
                 │
                 ▼
    ┌───────────────────────────────┐
    │ Step 3: Replace missing values │
    │ of selected variable           │
    └────────────┬──────────────────┘
                 │
                 ▼
    ┌───────────────────────────────┐
    │ Step 4: Move to next variable  │
    │ (e.g., Income → Score → Age…) │
    └────────────┬──────────────────┘
                 │
                 ▼
    ┌───────────────────────────────┐
    │ Step 5: Repeat the cycle until │
    │ results converge (stabilize)   │
    └────────────┬──────────────────┘
                 │
                 ▼
    ┌───────────────────────────────┐
    │ Step 6: Perform multiple runs  │
    │ (to handle randomness & bias)  │
    └────────────┬──────────────────┘
                 │
                 ▼
    ┌───────────────────────────────┐
    │ Final Multiple Imputed Datasets│
    │ → Combine results using Rubin's│
    │   Rules for final analysis     │
    └───────────────────────────────┘


# 🧩 Multivariate Imputation by Chained Equations (MICE)

**MICE (Multivariate Imputation by Chained Equations)** is a powerful method for filling missing data by modeling each incomplete variable *conditional on the others* and iterating the process until results stabilize.

---

## ⚙️ Step 1: Original Dataset (with Missing Values)

| Row | Age | Income | Score |
|------|------|---------|--------|
| 1 | 25 | 50000 | 3.5 |
| 2 | 30 | 60000 | NaN |
| 3 | NaN | 55000 | 4.0 |
| 4 | 40 | NaN | 4.5 |
| 5 | 35 | 65000 | 4.2 |

We have missing values in **Age**, **Income**, and **Score**.

---

## ⚙️ Step 2: Initial Imputation

We first fill missing values with **simple estimates** (like mean or median) just to start the chain.

- Mean Age = (25 + 30 + 40 + 35) / 4 = **32.5**  
- Mean Income = (50000 + 60000 + 55000 + 65000) / 4 = **57500**  
- Mean Score = (3.5 + 4.0 + 4.5 + 4.2) / 4 = **4.05**

| Row | Age | Income | Score |
|------|------|---------|--------|
| 1 | 25 | 50000 | 3.5 |
| 2 | 30 | 60000 | **4.05** |
| 3 | **32.5** | 55000 | 4.0 |
| 4 | 40 | **57500** | 4.5 |
| 5 | 35 | 65000 | 4.2 |

---

## 🔁 Step 3: Impute One Variable at a Time

We now start imputing **each variable** using regression models built on the others.

---

### (a) Imputing `Age`

We model `Age` based on `Income` and `Score` where `Age` is known.

| Row | Age | Income | Score |
|------|------|---------|--------|
| 1 | 25 | 50000 | 3.5 |
| 2 | 30 | 60000 | 4.05 |
| 4 | 40 | 57500 | 4.5 |
| 5 | 35 | 65000 | 4.2 |

Model:
$$
\text{Age} = 0.0003 \times \text{Income} + 5 \times \text{Score} - 60
$$

Predict missing `Age` for Row 3:
$$
\text{Age}_3 = 0.0003(55000) + 5(4.0) - 60 = 36.5
$$

Updated dataset:

| Row | Age | Income | Score |
|------|------|---------|--------|
| 1 | 25 | 50000 | 3.5 |
| 2 | 30 | 60000 | 4.05 |
| 3 | **36.5** | 55000 | 4.0 |
| 4 | 40 | 57500 | 4.5 |
| 5 | 35 | 65000 | 4.2 |

---

### (b) Imputing `Income`

We model `Income` using `Age` and `Score` where Income is known.

| Row | Age | Income | Score |
|------|------|---------|--------|
| 1 | 25 | 50000 | 3.5 |
| 2 | 30 | 60000 | 4.05 |
| 3 | 36.5 | 55000 | 4.0 |
| 5 | 35 | 65000 | 4.2 |

Model:
$$
\text{Income} = 2000 \times \text{Score} + 1000 \times \text{Age} - 15000
$$

Predict missing `Income` for Row 4:
$$
\text{Income}_4 = 2000(4.5) + 1000(40) - 15000 = 34000
$$

Updated dataset:

| Row | Age | Income | Score |
|------|------|---------|--------|
| 1 | 25 | 50000 | 3.5 |
| 2 | 30 | 60000 | 4.05 |
| 3 | 36.5 | 55000 | 4.0 |
| 4 | 40 | **34000** | 4.5 |
| 5 | 35 | 65000 | 4.2 |

---

### (c) Imputing `Score`

We model `Score` using `Age` and `Income` where Score is known.

| Row | Age | Income | Score |
|------|------|---------|--------|
| 1 | 25 | 50000 | 3.5 |
| 3 | 36.5 | 55000 | 4.0 |
| 4 | 40 | 34000 | 4.5 |
| 5 | 35 | 65000 | 4.2 |

Model:
$$
\text{Score} = 0.00001 \times \text{Income} + 0.02 \times \text{Age} - 0.5
$$

Predict missing `Score` for Row 2:
$$
\text{Score}_2 = 0.00001(60000) + 0.02(30) - 0.5 = 0.7
$$

Updated dataset:

| Row | Age | Income | Score |
|------|------|---------|--------|
| 1 | 25 | 50000 | 3.5 |
| 2 | 30 | 60000 | **0.7** |
| 3 | 36.5 | 55000 | 4.0 |
| 4 | 40 | 34000 | 4.5 |
| 5 | 35 | 65000 | 4.2 |

---

## 🔄 Step 4: Repeat the Chain

After one full cycle (`Age → Income → Score`), we **repeat the process** several times (5–20 iterations).  
Each iteration uses the latest values, gradually improving accuracy until **convergence**.

---

## 🧮 Step 5: Multiple Imputation

To handle randomness and uncertainty, we run the whole chained process **multiple times** (e.g., 5 datasets) with slightly different random seeds.

Then we:
- Analyze each imputed dataset separately.
- **Pool the results** using **Rubin’s rules** for final combined statistics.

---

## ✅ Summary Table

| Step | Operation | Target Variable | Technique | Result |
|------|-------------|----------------|------------|---------|
| 1 | Initialization | — | Fill with mean | Placeholder values |
| 2 | Imputation 1 | Age | Regression on Income & Score | Replace missing Age |
| 3 | Imputation 2 | Income | Regression on Age & Score | Replace missing Income |
| 4 | Imputation 3 | Score | Regression on Age & Income | Replace missing Score |
| 5 | Repeat | — | Iterate until stable | Converged imputations |
| 6 | Multiple Runs | — | Randomized sampling | Final multiple datasets |

---

## 💡 Summary

- **MICE** models each missing variable using the others in a chain.
- The process is **iterative** and continues until imputations stabilize.
- It can handle both **continuous and categorical** variables (with proper models).
- Produces **multiple imputations** to reflect uncertainty.
- Works best when data are **Missing At Random (MAR)**.

---



### Example with dataset

In [94]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.experimental import enable_iterative_imputer

In [95]:
from sklearn.impute import IterativeImputer

In [96]:
df = pd.DataFrame({
    'age': [25 , 27 , 29 , 33 , 33 , np.nan] ,
    'experience': [np.nan , 3 , 5 , 7 , 9 , 11],
    'salary': [50 , np.nan , 110 , 140 , 170 , 200],
    'purchased': [0 , 1 , 1  , 0, 1 , 0]
})
df

,age,experience,salary,purchased
0,25.0,NaN,50.0,0
1,27.0,3.0,NaN,1
2,29.0,5.0,110.0,1
3,33.0,7.0,140.0,0
4,33.0,9.0,170.0,1
5,NaN,11.0,200.0,0


In [97]:
x = df.drop(columns=['purchased'])
y = df['purchased']

In [98]:
x

,age,experience,salary
0,25.0,NaN,50.0
1,27.0,3.0,NaN
2,29.0,5.0,110.0
3,33.0,7.0,140.0
4,33.0,9.0,170.0
5,NaN,11.0,200.0


In [99]:
lr = LinearRegression()
imp = IterativeImputer(estimator=lr , verbose=2 , max_iter=10 , tol=1e-4 , imputation_order='roman')

In [100]:
imp.fit_transform(x)

[IterativeImputer] Completing matrix with shape (6, 3)
[IterativeImputer] Ending imputation round 1/10, elapsed time 0.01
[IterativeImputer] Change: 54.224236593551254, scaled tolerance: 0.02 
[IterativeImputer] Ending imputation round 2/10, elapsed time 0.01
[IterativeImputer] Change: 1.572682998439788, scaled tolerance: 0.02 
[IterativeImputer] Ending imputation round 3/10, elapsed time 0.02
[IterativeImputer] Change: 0.05553289948920792, scaled tolerance: 0.02 
[IterativeImputer] Ending imputation round 4/10, elapsed time 0.03
[IterativeImputer] Change: 0.025212949414822106, scaled tolerance: 0.02 
[IterativeImputer] Ending imputation round 5/10, elapsed time 0.03
[IterativeImputer] Change: 0.011441874497748472, scaled tolerance: 0.02 
[IterativeImputer] Early stopping criterion reached.


array([[ 25.        ,   1.00113226,  50.        ],
       [ 27.        ,   3.        ,  79.99049849],
       [ 29.        ,   5.        , 110.        ],
       [ 33.        ,   7.        , 140.        ],
       [ 33.        ,   9.        , 170.        ],
       [ 35.70252633,  11.        , 200.        ]])

In [101]:
x.corr()

,age,experience,salary
age,1.000000,0.946729,0.96833
experience,0.946729,1.000000,1.00000
salary,0.968330,1.000000,1.00000


**If we want to use only limited feature among large feature then we evaluate from correlation method . If features are highly corelated with target feature then it will be selected . It is significantly fast technique otherwise by default all feature are selected for imputation**

In [102]:
corr_values = [0.9 , 0.5 , 0.8 , 0.4 , 0.1]
corr_values

[0.9, 0.5, 0.8, 0.4, 0.1]

In [103]:
np.sum(corr_values)

np.float64(2.7)

In [104]:
0.9/2.7


0.3333333333333333

In [105]:
# Normalizing corelation values
from sklearn.preprocessing import normalize
probs = normalize([corr_values] ,norm='l1' )
prob = probs.ravel()
probs

array([[0.33333333, 0.18518519, 0.2962963 , 0.14814815, 0.03703704]])

In [106]:
np.random.choice(['var1', 'var2', 'var3', 'var4', 'var5'], 2 , replace=False ,p=prob)  # Higher corr = higher chance of selection

array(['var4', 'var1'], dtype='<U4')

Imputation in test-set also

In [107]:
df = pd.DataFrame({
    'age': [25, 27, 29, 33, 33, np.nan , 37 , 39 , 41,  np.nan , 45],
    'experience': [np.nan, 3, 5, 7, 9, 11 , 13 , 16 , np.nan , 19 , 21],
    'salary': [50, np.nan, 110, 140, 170, 200, 230 , 260 , np.nan , 320 , 350],
    'purchased': [0, 1, 1, 0, 1, 0 , 0 , 1 , 1 , 0 , 0]
})
df

,age,experience,salary,purchased
0,25.0,NaN,50.0,0
1,27.0,3.0,NaN,1
2,29.0,5.0,110.0,1
3,33.0,7.0,140.0,0
4,33.0,9.0,170.0,1
5,NaN,11.0,200.0,0
6,37.0,13.0,230.0,0
7,39.0,16.0,260.0,1
8,41.0,NaN,NaN,1
9,NaN,19.0,320.0,0


In [108]:
x = df.drop(columns=['purchased'])
y = df['purchased']

In [109]:
from sklearn.model_selection import train_test_split
x_train , x_test , y_train  , y_test = train_test_split(x , y , test_size=0.2 , shuffle=False)

In [110]:
x_train

,age,experience,salary
0,25.0,NaN,50.0
1,27.0,3.0,NaN
2,29.0,5.0,110.0
3,33.0,7.0,140.0
4,33.0,9.0,170.0
5,NaN,11.0,200.0
6,37.0,13.0,230.0
7,39.0,16.0,260.0


In [111]:
x_train.mean()

age            31.857143
experience      9.142857
salary        165.714286
dtype: float64

In [112]:
x_test

,age,experience,salary
8,41.0,NaN,NaN
9,NaN,19.0,320.0
10,45.0,21.0,350.0


In [113]:
imp.imputation_sequence_

[_ImputerTriplet(feat_idx=np.int64(0), neighbor_feat_idx=array([1, 2]), estimator=LinearRegression()),
 _ImputerTriplet(feat_idx=np.int64(1), neighbor_feat_idx=array([0, 2]), estimator=LinearRegression()),
 _ImputerTriplet(feat_idx=np.int64(2), neighbor_feat_idx=array([0, 1]), estimator=LinearRegression()),
 _ImputerTriplet(feat_idx=np.int64(0), neighbor_feat_idx=array([1, 2]), estimator=LinearRegression()),
 _ImputerTriplet(feat_idx=np.int64(1), neighbor_feat_idx=array([0, 2]), estimator=LinearRegression()),
 _ImputerTriplet(feat_idx=np.int64(2), neighbor_feat_idx=array([0, 1]), estimator=LinearRegression()),
 _ImputerTriplet(feat_idx=np.int64(0), neighbor_feat_idx=array([1, 2]), estimator=LinearRegression()),
 _ImputerTriplet(feat_idx=np.int64(1), neighbor_feat_idx=array([0, 2]), estimator=LinearRegression()),
 _ImputerTriplet(feat_idx=np.int64(2), neighbor_feat_idx=array([0, 1]), estimator=LinearRegression()),
 _ImputerTriplet(feat_idx=np.int64(0), neighbor_feat_idx=array([1, 2]), e

In [114]:
imp.transform(x_test)

[IterativeImputer] Completing matrix with shape (3, 3)
[IterativeImputer] Ending imputation round 1/5, elapsed time 0.00
[IterativeImputer] Ending imputation round 2/5, elapsed time 0.00
[IterativeImputer] Ending imputation round 3/5, elapsed time 0.00
[IterativeImputer] Ending imputation round 4/5, elapsed time 0.00
[IterativeImputer] Ending imputation round 5/5, elapsed time 0.00


array([[ 41.        ,  12.85376394, 227.81648435],
       [ 43.83428599,  19.        , 320.        ],
       [ 45.        ,  21.        , 350.        ]])